In [4]:
vocab = '$abcdefghijklmnopqrstuvwxyz'
vocab_size = len(vocab)
ch_to_i = {char: i for i, char in enumerate(vocab)}
i_to_ch = {i: char for i, char in enumerate(vocab)}


import torch
import torch.nn as nn
import torch.nn.functional as F

equal_probs = F.softmax(torch.ones(vocab_size), dim=0)
for i in range(5):
    generated = ""
    while True:
        random_int = torch.multinomial(equal_probs, 1).item()
        random_char = i_to_ch[random_int]
        if random_char == "$":
            break
        generated += random_char
    print(f"name {i}: {generated}")

name 0: biasvtgrdmhvacsgdvfao
name 1: xusunatopbwcjkx
name 2: foqydjkyglgprlwaf
name 3: bjhhcwbsfxjhctnbcvewn
name 4: lneli


In [5]:
# Self-supervised learning
names = []
with open("../../data/yob2025.txt", 'r') as file:
    for line in file:
        name, _, _ = line.lower().strip().split(',')
        names.append("$" + name + "$")
len(names)

31227

In [7]:
bigram = torch.zeros((vocab_size, vocab_size))
total = 0
for name in names:
    for ch1, ch2 in zip(name, name[1:]):
        ch1_int = ch_to_i[ch1]
        ch2_int = ch_to_i[ch2]
        bigram[ch1_int][ch2_int] += 1
        total += 1
bigram /= total

for i in range(5):
    generated = "$"
    while True:
        bigram_probs = bigram[ch_to_i[generated[-1]]]
        sampled_char = i_to_ch[torch.multinomial(bigram_probs, 1).item()]
        if sampled_char == "$":
            break
        generated += sampled_char
    print(f"name {i}: {generated[1:]}")
        

name 0: coslis
name 1: gre
name 2: zaydasatal
name 3: try
name 4: janive


In [8]:
example_name = "$ada$"
encode = lambda word: torch.tensor([ch_to_i[c] for c in word])
decode = lambda tensor_i: "".join(i_to_ch[i.item()] for i in tensor_i)
print(encode(example_name))
print(decode(encode(example_name)))

tensor([0, 1, 4, 1, 0])
$ada$


In [10]:
name_indices = [encode(name) for name in names]
target_indices = [name_index[1:] for name_index in name_indices]

# Pad the input and target sequences to the same length to facilitate processing for out models
from torch.nn.utils.rnn import pad_sequence
X = pad_sequence(name_indices, batch_first=True, padding_value=0)
max_name_length = max(len(name) for name in names)

target_indices.append(torch.empty((max_name_length), dtype=torch.long))

Y = pad_sequence(target_indices, batch_first=True, padding_value=-1)[:-1]
print(X[0])
print(Y[0])

tensor([ 0, 15, 12,  9, 22,  9,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
tensor([15, 12,  9, 22,  9,  1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])


In [12]:
X.size()

torch.Size([31227, 17])

In [13]:
def get_batch(batch_size=64):
    random_index = torch.randint(0, X.size(0), (batch_size, ))
    inputs = X[random_index]
    labels = Y[random_index]
    return inputs, labels

inputs, labels = get_batch(3)
print(inputs)
print(labels)

tensor([[ 0, 13, 25,  1, 18,  9,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  1, 13, 25, 18,  9, 19,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0, 19, 21,  2,  1, 18, 21,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])
tensor([[13, 25,  1, 18,  9,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [ 1, 13, 25, 18,  9, 19,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
        [19, 21,  2,  1, 18, 21,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]])


In [14]:
embedding_dim = 3
embedding = nn.Embedding(vocab_size, embedding_dim)
example_input = torch.tensor([1,1,0,2])
input_embd = embedding(example_input)
print(input_embd.shape)
input_embd

torch.Size([4, 3])


tensor([[-0.2067,  0.2281,  0.7737],
        [-0.2067,  0.2281,  0.7737],
        [ 1.7712,  0.8696,  1.6063],
        [-0.4760,  1.5474, -1.9210]], grad_fn=<EmbeddingBackward0>)